In [28]:
# import libraries
import numpy as np
import scipy as sp
import pickle
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import concurrent.futures
import os

In [29]:
# Categorization functions

# def categorize_diffusers(coordinates):
#     """Recieves a coordinates array of shape [n_diffusers, 3], and returns an array of shape [n_diffusers],
#     Where each entry is in the range [0,9] (inclusive). Which signify the following state states:
#     0: Cytoplasm
#     1-8: NPC radial segmets
#     9: Nucleus
#     """
#     n_diffusers = coordinates.shape[0]
#     categorized_coordinates = np.zeros(shape=(n_diffusers))
    
#     # Create masks
#     cyto_mask = coordinates[:, 2] >= 15
#     nuc_mask = coordinates[:, 2] <= -15
    
#     # apply masks
#     angle = np.arctan2(coordinates[:, 1], coordinates[:, 0]) + np.pi
#     eighth = np.floor(angle / (np.pi * 0.25)) + 1
#     eighth[eighth == 9] = 8
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask))] = eighth[((~cyto_mask) & (~nuc_mask))]
#     categorized_coordinates[cyto_mask] = 0
#     categorized_coordinates[nuc_mask] = 9
    
#     return categorized_coordinates


# def categorize_diffusers2(coordinates, angle_offset=0):
#     """
#     Recieves a coordinates array of shape [n_diffusers, 3], and returns an array of shape [n_diffusers],
#     Where each entry is in the range [0,17] (inclusive). Which signify the following state states:
#     0: Cytoplasm
#     1-8: NPC radial segmets (cytoplasm half)
#     9-16: NPC radial segmets (nucleus half)
#     17: Nucleus
#     angle_offset should be a number between 0 and 0.25*pi, signifying the angle offset for spoke barriers.
#     """
#     n_diffusers = coordinates.shape[0]
#     categorized_coordinates = np.zeros(shape=(n_diffusers))
    
#     # Create masks
#     cyto_mask = coordinates[:, 2] >= 15
#     nuc_mask = coordinates[:, 2] <= -15
#     cyto_half_mask = coordinates[:, 2] >= 0
    
#     # apply masks
#     angle = np.arctan2(coordinates[:, 1], coordinates[:, 0]) + np.pi + angle_offset
#     angle = np.mod(angle, 2 * np.pi)
#     eighth = np.floor(angle / (np.pi * 0.25)) + 1
#     eighth[eighth == 9] = 8
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & cyto_half_mask)] = eighth[((~cyto_mask) & (~nuc_mask) & cyto_half_mask)]
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & (~cyto_half_mask))] = eighth[((~cyto_mask) & (~nuc_mask) & (~cyto_half_mask))] + 8
#     categorized_coordinates[cyto_mask] = 0
#     categorized_coordinates[nuc_mask] = 17
    
#     return categorized_coordinates

# def categorize_diffusers3(coordinates, angle_offset=0):
#     """
#     Recieves a coordinates array of shape [n_diffusers, 3], and returns an array of shape [n_diffusers],
#     Where each entry is in the range [0,25] (inclusive). Which signify the following state states:
#     0: Cytoplasm
#     1-8: NPC radial segmets (cytoplasm third)
#     9-16: NPC radial segments (center third)
#     17-24: NPC radial segmets (nucleus third)
#     25: Nucleus
#     angle_offset should be a number between 0 and 0.25*pi, signifying the angle offset for spoke barriers.
#     """
#     n_diffusers = coordinates.shape[0]
#     categorized_coordinates = np.zeros(shape=(n_diffusers))
    
#     # Create masks
#     cyto_mask = coordinates[:, 2] >= 15
#     nuc_mask = coordinates[:, 2] <= -15
#     cyto_third_mask = coordinates[:, 2] >= 5
#     nuc_third_mask = coordinates[:, 2] <= -5
#     center_third_mask = ~cyto_third_mask & ~nuc_third_mask
    
    
#     # apply masks
#     angle = np.arctan2(coordinates[:, 1], coordinates[:, 0]) + np.pi + angle_offset
#     angle = np.mod(angle, 2 * np.pi)
#     eighth = np.floor(angle / (np.pi * 0.25)) + 1
#     eighth[eighth == 9] = 8
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & cyto_third_mask)] = eighth[((~cyto_mask) & (~nuc_mask) & cyto_third_mask)]
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & center_third_mask)] = eighth[((~cyto_mask) & (~nuc_mask) & center_third_mask)] + 8
#     categorized_coordinates[((~cyto_mask) & (~nuc_mask) & nuc_third_mask)] = eighth[((~cyto_mask) & (~nuc_mask) & nuc_third_mask)] + 16
#     categorized_coordinates[cyto_mask] = 0
#     categorized_coordinates[nuc_mask] = 25
    
#     return categorized_coordinates

def categorize_diffusers_n(coordinates, n, angle_offset=0):
    """
    Recieves a coordinates array of shape [n_diffusers, 3], and returns an array of shape [n_diffusers],
    Where each entry is in the range [0,n*8 + 1] (inclusive). Which signify the following state states:
    0: Cytoplasm
    i+1-i+9: 8 NPC radial segmets in the i'th vertical layer
    n*8 + 1: Nucleus
    angle_offset should be a number between 0 and 0.25*pi, signifying the angle offset for spoke barriers.
    """
    n_diffusers = coordinates.shape[0]
    categorized_coordinates = np.zeros(shape=(n_diffusers))
    
    # Create masks
    cyto_mask = coordinates[:, 2] >= 15
    nuc_mask = coordinates[:, 2] <= -15
    
    # Calculate the vertical layer boundaries
    layer_size = 30 / n  # 30 is the total height (-15 to 15)
    layer_boundaries = np.linspace(-15, 15, n+1)
    
    # Calculate angles for radial segmentation
    angle = np.arctan2(coordinates[:, 1], coordinates[:, 0]) + np.pi + angle_offset
    angle = np.mod(angle, 2 * np.pi)
    eighth = np.floor(angle / (np.pi * 0.25)) + 1
    eighth[eighth == 9] = 8  # Handle edge case
    
    # Apply cytoplasm and nucleus masks first
    categorized_coordinates[cyto_mask] = 0
    categorized_coordinates[nuc_mask] = n*8 + 1
    
    # For each vertical layer, categorize the diffusers
    for i in range(n):
        layer_mask = (~cyto_mask & ~nuc_mask & 
                      (coordinates[:, 2] >= layer_boundaries[i]) & 
                      (coordinates[:, 2] < layer_boundaries[i+1]))
        
        # Assign values for this layer: i*8 + eighth (with offset adjustment)
        categorized_coordinates[layer_mask] = i*8 + eighth[layer_mask]
    
    return categorized_coordinates

    
def categorize_diffusers_over_time(trajectories, angle_offset, step=1, n_layers=1):
    n_diffusers = trajectories.shape[0]
    n_t = trajectories.shape[2]
    n_columns = len(range(0, n_t, step))
    categorized_trajectories = np.zeros(shape=(n_diffusers, n_columns))
    for t in range(0, n_t, step):
        categorized_trajectories[:,int(t / step)] = categorize_diffusers_n(trajectories[:, :, t], n_layers, angle_offset=angle_offset)
    return categorized_trajectories, n_layers * 8 + 2

In [30]:
# Transition matrix functions

from numpy import float64


def normalize_rows(counts_matrix):
    transition_matrix = np.zeros_like(counts_matrix)
    n_states = counts_matrix.shape[0]
    for i in range(n_states):
        row_sum = np.sum(counts_matrix[i, :])
        if row_sum == 0:
            print(f"Row sum is 0, setting row {i} to 0    :( ")
            transition_matrix[i, :] = 0
            continue
        transition_matrix[i, :] = counts_matrix[i, :] / row_sum
    return transition_matrix
    

def calc_counts_matrix(categorized_trajectories, n_states, init_1 = False):
    if init_1: counts_matrix = np.ones(shape=(n_states,n_states))
    else: counts_matrix = np.zeros(shape=(n_states,n_states))
    n_diffusers = categorized_trajectories.shape[0]
    n_t = categorized_trajectories.shape[1]
    
    for i in range(n_diffusers):
        for t in range(n_t - 1):
            first = int(categorized_trajectories[i,t])
            second = int(categorized_trajectories[i,t+1])
            counts_matrix[first, second] += 1
    return counts_matrix

def eightwise_symmetrize(data):
    """Given a matrix, makes it 8-wise symmetric"""
    if (data.shape[0] % 8 != 0) or (data.shape[1] % 8 != 0):
        raise Exception("invalid matrix shape")
    mask_base = np.array(np.eye(8, dtype=bool))
    masks = [np.roll(mask_base, shift=i, axis=0) for i in range(8)]
    new_data = np.zeros_like(data)
    for mask in masks:
        for i in range(int(data.shape[0] / 8)):
            for j in range(int(data.shape[1] / 8)):
                new_data[i*8:(i+1)*8, j*8:(j+1)*8][mask] += np.sum(data[i*8:(i+1)*8, j*8:(j+1)*8][mask])
    return new_data

def calc_transition_matrix(categorized_trajectories, n_states, symmetrize=False, init_1 = False):
    counts_matrix = calc_counts_matrix(categorized_trajectories, n_states, init_1=init_1)
    if symmetrize: counts_matrix[1:-1, 1:-1] = eightwise_symmetrize(counts_matrix[1:-1, 1:-1])
    transition_matrix = normalize_rows(counts_matrix)
    return transition_matrix


In [ ]:
# Process data
categorized_trajectories, n_states = categorize_diffusers_over_time(trajectories=trajectories,
                                                                    angle_offset=3,
                                                                    n_layers=10)
tm = calc_transition_matrix(categorized_trajectories=categorized_trajectories,
                            n_states=n_states,
                            symmetrize=True,
                            init_1=True)
with open("data/transition_matrices/spatial-150-180-symmetric-10layers.pickle", "wb") as f:
    pickle.dump(tm, f)

In [31]:
def count_transports(sim):
    sim = sim.tolist()
    transports = 0
    cur = -1
    for i in range(len(sim)-1):
        if cur == -1:
            if sim[i+1] == 0 or sim[i+1] == 0:
                cur = sim[i+1]
                continue
        elif cur == 0 and sim[i+1] == 9:
            transports += 1
            cur = 9
        elif cur == 9 and sim[i+1] == 0:
            transports += 1
            cur = 0
    return transports

def count_transport_events_multiple( sim_indexes,
                            sim_times,
                            step=1,
                            load_path_prefix="data/singles/"):
    #############
    # Load data #
    #############
    i_time_iterator = [(i, time) for i in sim_indexes for time in sim_times]

    def process_file(i_time):
        i, time = i_time
        print(f"{time}, {i} ", end="")
        with open(f"{load_path_prefix}/{i}/{time}.pickle", "rb") as f:
            diffuser_trajectories = pickle.load(f)
        return categorize_diffusers_over_time(trajectories=diffuser_trajectories,
                                              angle_offset=0,
                                              step=step,
                                              n_layers=1)[0]

    arrays = []
    num_processes = len(os.sched_getaffinity(0))
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
        results = list(executor.map(process_file, i_time_iterator))

    arrays.extend(results)
    all = np.concatenate(arrays, axis=0)
    
    transports = 0
    for i in range(all.shape[0]):
        transports += count_transports(all[i,:])
    print(transports)
    return transports

In [41]:
count_transport_events_multiple(
sim_indexes = range(1, 51),
sim_times = ["150-180"],
step = 1,
load_path_prefix="data/singles"
)

150-180, 1 150-180, 2 150-180, 3 150-180, 4 150-180, 5 150-180, 6 150-180, 7 150-180, 8 150-180, 9 150-180, 10 150-180, 11 150-180, 12 150-180, 13 150-180, 14 150-180, 15 150-180, 16 150-180, 17 150-180, 18 150-180, 19 150-180, 20 150-180, 21 150-180, 22 150-180, 23 150-180, 24 150-180, 25 150-180, 26 150-180, 27 150-180, 28 150-180, 29 150-180, 30 150-180, 31 150-180, 32 150-180, 33 150-180, 34 150-180, 35 150-180, 36 150-180, 37 150-180, 38 150-180, 39 150-180, 40 150-180, 41 150-180, 42 150-180, 43 150-180, 44 150-180, 45 150-180, 46 150-180, 47 150-180, 48 150-180, 49 150-180, 50 26


26